# Sentiment Analysis - Minecraft Reviews (TensorFlow)
**Nama**: Asep Haryana Saputra

## Project Overview
**Objective**: Build a high-accuracy (>92%) sentiment analysis model.
**Dataset**: 40,000 Minecraft reviews.
**Framework**: TensorFlow / Keras.

## 1. Import Libraries

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import tensorflow as tf
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
try:
    from tensorflow.keras.layers import TextVectorization, SpatialDropout1D, BatchNormalization, Conv1D, GlobalMaxPooling1D
except ImportError:
    from tensorflow.keras.layers.experimental.preprocessing import TextVectorization
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, GRU, Dropout, GlobalAveragePooling1D, Input, SpatialDropout1D, BatchNormalization, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

# Optimize GPU Memory
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


TensorFlow Version: 2.19.0


In [17]:
# Callbacks Definition
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

def get_callbacks(model_name):
    checkpoint = ModelCheckpoint(
        f'{model_name}_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        min_delta=0.001,
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    return [checkpoint, early_stopping, reduce_lr]

## 2. Data Loading & Preprocessing
**Labeling Strategy**:
- 1-2 Stars -> **Negative (0)**
- 3 Stars -> **Neutral (1)**
- 4-5 Stars -> **Positive (2)**

In [18]:
df = pd.read_csv('/content/dataset_raw.csv')

# Create 'label' column based on 'score'
def get_sentiment_label(score):
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['score'].apply(get_sentiment_label)
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,label
0,ef71e46c-798f-40aa-b1f3-09f30e43da40,robert seal,https://play-lh.googleusercontent.com/a-/ALV-U...,it's really fun but I hardly get to play with ...,5,0,1.21.132.1,2026-02-08 22:21:14,NaN,NaN,1.21.132.1,2
1,b26157d7-a1c6-4236-a7e1-770c4a819a61,Jaynard jake Urbano narag,https://play-lh.googleusercontent.com/a-/ALV-U...,add more blocks in creative please 🙏🥺,5,0,1.21.132.1,2026-02-08 22:20:06,NaN,NaN,1.21.132.1,2
2,7b361865-5743-4ac7-bf4c-e1274ddda36c,Adrian Josh Gamuzaran,https://play-lh.googleusercontent.com/a-/ALV-U...,Bring back multiplayer via hotspot! OFFLINE MU...,3,0,NaN,2026-02-08 22:19:12,NaN,NaN,NaN,1
3,ad9f2866-6db0-482d-813c-d1cc2f2daf12,Sulaiman Jasim,https://play-lh.googleusercontent.com/a/ACg8oc...,this game improve our creativity,5,0,NaN,2026-02-08 22:07:31,NaN,NaN,NaN,2
4,565b3482-f695-4f41-bfda-0f9fad4e7aca,Florin Varga,https://play-lh.googleusercontent.com/a/ACg8oc...,i love this game but it severly underutilises ...,3,0,1.21.132.1,2026-02-08 22:01:11,NaN,NaN,1.21.132.1,1


In [ ]:
# Advanced Data Cleaning (Lemmatization + POS Tagging)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Download necessary NLTK data
nltk_packages = ['stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4', 'punkt']
for pkg in nltk_packages:
    try:
        if pkg == 'punkt':
             nltk.data.find('tokenizers/punkt')
        elif pkg in ['stopwords', 'wordnet', 'omw-1.4']:
             nltk.data.find(f'corpora/{pkg}')
        else:
             nltk.data.find(f'taggers/{pkg}')
    except LookupError:
        nltk.download(pkg)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Explicitly keep negation words AND conjunctions like "but" for nuance
negation_words = {'no', 'not', 'nor', 'neither', 'none', 'never', "don't", "can't", "won't", "isn't", "aren't"}
context_words = {'but', 'however', 'although'} # Pivotal words for sentiment shift
stop_words = stop_words - negation_words - context_words

# Contraction Map (Standard English)
contraction_map = {
    "don't": "do not", "can't": "cannot", "won't": "will not",
    "isn't": "is not", "aren't": "are not", "didn't": "did not",
    "doesn't": "does not", "haven't": "have not", "hadn't": "had not",
    "wouldn't": "would not", "couldn't": "could not", "shouldn't": "should not",
    "it's": "it is", "i'm": "i am", "you're": "you are", "we're": "we are", "they're": "they are"
}

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean_text(text):
    text = str(text).lower()
    
    # 1. Expand Basic Contractions
    for word, replacement in contraction_map.items():
        text = text.replace(word, replacement)
    
    # 2. Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # 3. Simple Regex (Keep alphanumeric + basic punctuation)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text) # Replace special chars with space
    
    # 4. Tokenization & POS Tagging
    if not isinstance(text, str): text = str(text)
    
    tokens = text.split() # Fallback to split for speed/robustness
    if not tokens: return ""
    
    pos_tags = nltk.pos_tag(tokens)
    
    # 5. Lemmatization with POS & Stopword Removal
    cleaned_tokens = []
    for word, tag in pos_tags:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
            cleaned_tokens.append(lemma)
            
    return " ".join(cleaned_tokens)

# Test the cleaner
print("Original: The game is good but it crashes.")
print("Cleaned :", clean_text("The game is good but it crashes."))



In [ ]:
# 3. Data Quality Filtering & Splitting
if 'content' in df.columns:
    # 1. Remove Timestamp Rows (Specific User Feedback)
    # Pattern matches exactly HH:MM:SS
    timestamp_pattern = r'^\\d{2}:\\d{2}:\\d{2}$'
    df_clean = df[~df['content'].astype(str).str.match(timestamp_pattern)].copy()
    
    # 2. Remove Short Rows (< 5 chars)
    # User suggested dropping "👍" loops or super short content
    df_clean = df_clean[df_clean['content'].astype(str).str.len() >= 5].copy()
    
    # 3. Remove other basic constraints (keep existing word_count >= 3 logic if useful, 
    # but len >= 5 likely covers it)
    df_clean['word_count'] = df_clean['content'].apply(lambda x: len(str(x).split()))
    df_clean = df_clean[df_clean['word_count'] >= 3] # Keep existing safeguard
    
    # Optional: Thumbs Up Weighting (for future use)
    # df_clean['sample_weight'] = np.log1p(df_clean['thumbsUpCount']) 

    print(f"Original shape: {df.shape}, Cleaned shape: {df_clean.shape}")

    X = df_clean['content'].values
    y = df_clean['label'].values

    # Scheme 1 & 3: 80/20 Split
    X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Scheme 2: 90/10 Split
    X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

    print(f"Scheme 1/3 (80/20) - Train: {X_train_80.shape}, Test: {X_test_80.shape}")
    print(f"Scheme 2 (90/10)   - Train: {X_train_90.shape}, Test: {X_test_90.shape}")
else:
    print("Error: 'content' column not found yet. Ensure Data Loading cell is run first.")



## 3. Tokenization & Splitting

In [21]:
# Tokenization for Scheme 1 & 2 (Embedding Models)
VOCAB_SIZE = 10000
MAX_LENGTH = 100
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

# Prepare Sequences for Scheme 1 (80/20)
seq_train_80 = tokenizer.texts_to_sequences(X_train_80)
seq_test_80 = tokenizer.texts_to_sequences(X_test_80)
pad_train_80 = pad_sequences(seq_train_80, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_80 = pad_sequences(seq_test_80, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Prepare Sequences for Scheme 2 (90/10)
seq_train_90 = tokenizer.texts_to_sequences(X_train_90)
seq_test_90 = tokenizer.texts_to_sequences(X_test_90)
pad_train_90 = pad_sequences(seq_train_90, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_90 = pad_sequences(seq_test_90, maxlen=MAX_LENGTH, padding='post', truncating='post')

In [22]:
# Force CPU for TextVectorization to avoid GPU OOM
with tf.device('/CPU:0'):
    # Preprocessing for Scheme 3 (TF-IDF)
    vectorize_layer = TextVectorization(
        max_tokens=VOCAB_SIZE,
        output_mode='tf-idf',
        ngrams=2 # improved features
    )
    # Adapt on 80% train data
    vectorize_layer.adapt(X_train_80)

    tfidf_train_80 = vectorize_layer(X_train_80)
    tfidf_test_80 = vectorize_layer(X_test_80)


## 3.1 Load Pre-trained Embeddings


## 4. Experiment 1: Bi-LSTM Model

In [ ]:
# Scheme 1 (Baseline): Bi-LSTM + Embedding + 80/20 Split
tf.random.set_seed(42)
model1 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme1_BiLSTM')

model1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model1.summary()

history1 = model1.fit(
    pad_train_80, y_train_80,
    epochs=10,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme1_bilstm')
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "Scheme1_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7541 - loss: 0.7031
Epoch 1: val_accuracy improved from -inf to 0.84969, saving model to scheme1_bilstm_best.keras
819/819 ━━━━━━━━━━━━━━━━━━━━ 29s 24ms/step - accuracy: 0.7541 - loss: 0.7029 - val_accuracy: 0.8497 - val_loss: 0.5291 - learning_rate: 0.0010
Epoch 2/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8546 - loss: 0.4332
Epoch 2: val_accuracy improved from 0.84969 to 0.84984, saving model to scheme1_bilstm_best.keras
819/819 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.8546 - loss: 0.4332 - val_accuracy: 0.8498 - val_loss: 0.4966 - learning_rate: 0.0010
Epoch 3/50
818/819 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8805 - loss: 0.3566
Epoch 3: val_accuracy did not improve from 0.84984
819/819 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.8805 - loss: 0.3566 - val_accuracy: 0.8436 - val_loss: 0.5228 - learning_rate: 0.0010
Epoch 4/50
817/819 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0

## 5. Experiment 2: GRU Model

In [ ]:
# Scheme 2 (Data Variation): GRU + Embedding + 90/10 Split
tf.random.set_seed(42)
model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    GRU(128, return_sequences=True),
    Dropout(0.3),
    GRU(64),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme2_GRU_90_10')

model2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model2.summary()

history2 = model2.fit(
    pad_train_90, y_train_90,
    epochs=10,
    validation_data=(pad_test_90, y_test_90),
    batch_size=32,
    callbacks=get_callbacks('scheme2_gru')
)

## 6. Experiment 3: Simple NN (Baseline)

In [ ]:
# Scheme 3 (Feature Variation): Dense + TF-IDF + 80/20 Split
tf.random.set_seed(42)
model3 = Sequential([
    Input(shape=(VOCAB_SIZE,)), # TF-IDF input
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(3, activation='softmax')
], name='Scheme3_Dense_TFIDF')

model3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model3.summary()

history3 = model3.fit(
    tfidf_train_80, y_train_80,
    epochs=10,
    validation_data=(tfidf_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme3_tfidf')
)

## 8. Experiment 5: Conv1D (N-Gram Learning)


In [ ]:
# Scheme 5: Conv1D for N-Gram feature extraction
# Conv1D filters act as N-gram detectors (e.g. kernel_size=2 sees 2 words)
tf.random.set_seed(42)
model5 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Conv1D(128, 5, activation='relu'), # Detects 5-grams
    GlobalMaxPooling1D(), # Captures most important features
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme5_Conv1D_NGram')

model5.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model5.summary()

history5 = model5.fit(
    pad_train_80, y_train_80,
    epochs=50,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme5_conv1d')
)


## 7. Evaluation & Inference

In [ ]:
# Inference Function
def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)
    pred = model1.predict(padded)
    label_idx = np.argmax(pred)

    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return mapping[label_idx]

# Test
print(predict_sentiment("This game is super fun!"))
print(predict_sentiment("Too many bugs, unplayable."))